# Recipe Generator (Toy Project)

This notebook dynamically loads ingredient JSON files from `../data/`, builds a prompt from them, and calls an LLM deployed on **Azure AI Foundry** (Azure OpenAI) to generate a recipe.

**Purpose:** this is a toy repo meant for practicing a real dev workflow — branch from a Jira ticket, make a small change here (e.g. add a new ingredient category, tweak the prompt, add a dietary filter), commit, push, and open a PR.

Setup:
1. Copy `.env.example` to `.env` at the repo root and fill in your Azure AI Foundry endpoint, key, and deployment name.
2. `pip install -r requirements.txt`
3. Run the cells below.

In [ ]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("..").resolve()))  # so `src` is importable from notebooks/
from src.recipe_utils import build_prompt, call_llm, exclude_by_dietary, filter_by_dietary, load_ingredient_data, sample_ingredients, save_output

load_dotenv(Path("..") / ".env")  # reads .env at the repo root

DATA_DIR = Path("..") / "data"
OUTPUTS_DIR = Path("..") / "outputs"  # generated recipes are logged here as JSON

# Dietary requirement applied to the whole flow. Set to None for no restriction,
# or e.g. "vegan", "vegetarian", "pescatarian", "dairy-free" (see DIETARY_CONFLICTS
# in src/recipe_utils.py).
DIETARY = "vegan"

# How DIETARY is enforced when picking ingredients:
#   "exclude" — deny-list (exclude_by_dietary): keep everything except items whose
#               tags conflict, e.g. keep vegetables/grains for "vegan", drop meat & dairy.
#   "filter"  — allow-list (filter_by_dietary): keep ONLY items explicitly tagged with
#               DIETARY. Stricter — an untagged-but-harmless item (e.g. rice) is dropped.
DIETARY_MODE = "exclude"

## 1. Dynamically load every ingredient JSON file in `data/`

Any `.json` file dropped into the `data/` folder is picked up automatically — no code changes needed. Each file is expected to look like:
```json
{"category": "proteins", "items": [{"name": "chicken breast", "tags": ["poultry", "lean"]}]}
```

In [ ]:
# load_ingredient_data() lives in src/recipe_utils.py (unit tested in tests/test_recipe_utils.py)
ingredient_bank = load_ingredient_data(DATA_DIR)

for category, items in ingredient_bank.items():
    print(f"{category}: {len(items)} items")

## 2. Pick a random pool of ingredients

Grabs a few items per category to hand to the model. Swap this out for real pantry input, a UI, etc.

In [ ]:
# Narrow the ingredient bank to what's compatible with DIETARY before sampling,
# so the model never sees ingredients that violate the requirement. The two
# helpers (both in src/recipe_utils.py) take opposite approaches — pick via
# DIETARY_MODE above.
if not DIETARY:
    eligible = ingredient_bank
elif DIETARY_MODE == "filter":
    eligible = filter_by_dietary(ingredient_bank, DIETARY)  # allow-list
else:
    eligible = exclude_by_dietary(ingredient_bank, DIETARY)  # deny-list

chosen = sample_ingredients(eligible, per_category=2, seed=42)
print(json.dumps(chosen, indent=2))

## 3. Build the prompt

The sampled JSON is embedded directly into the prompt sent to the model.

In [ ]:
prompt = build_prompt(chosen, cuisine="Mediterranean", dietary=DIETARY)
print(prompt)

## 4. Call the LLM on Azure AI Foundry

In [ ]:
# get_client() and call_llm() also live in src/recipe_utils.py — imported above.
# (Not unit tested directly since they make a real network call to Azure; the
# testable logic — prompt building, sampling, filtering — is what tests/ covers.)
#
# call_llm() automatically retries on rate-limit errors (HTTP 429) with
# exponential backoff via the retry_on_rate_limit() wrapper in src/recipe_utils.py,
# so a transient 429 from Azure is retried rather than raised. The retry logic
# IS unit tested (with a fake client / mocked sleep) in tests/test_recipe_utils.py.

In [ ]:
recipe = call_llm(prompt)
print(recipe)

## 5. Log the run to `outputs/`

Save the chosen ingredients, the prompt, and the generated recipe to a timestamped JSON file so each run is reproducible and reviewable. `save_output()` lives in `src/recipe_utils.py` (unit tested in `tests/test_recipe_utils.py`).

In [ ]:
out_path = save_output(chosen, recipe, OUTPUTS_DIR, prompt=prompt)
print(f"Saved run to {out_path}")

## Ideas for a first "junior engineer" PR

Small, self-contained changes that exercise the branch → commit → push → PR workflow:

- Add a new `data/desserts.json` category and confirm it's picked up automatically.
- Add a `dietary` filter that excludes items whose `tags` conflict (e.g. exclude non-vegan items when `dietary="vegan"`).
- Add a retry/backoff wrapper around `call_llm` for rate-limit errors (and a test using a fake/mocked client).
- Extend `tests/test_recipe_utils.py` with a case for an empty `data/` folder or a malformed JSON file.
- Log the chosen ingredients + generated recipe to an `outputs/` folder as JSON.
- Wire up `filter_by_dietary` (already in `src/recipe_utils.py`) into the prompt-building flow in this notebook.